In [67]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

## 1. **EDA**, **Missing values treatment**, **Outlier treatment**
  


In [ ]:
df = pd.read_csv('cs.csv')

In [ ]:
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df)

                                      Missing Count  Missing %
Unnamed: 0                                        0   0.000000
SeriousDlqin2yrs                                  0   0.000000
RevolvingUtilizationOfUnsecuredLines              0   0.000000
age                                               0   0.000000
NumberOfTime30-59DaysPastDueNotWorse              0   0.000000
DebtRatio                                         0   0.000000
MonthlyIncome                                 29731  19.820667
NumberOfOpenCreditLinesAndLoans                   0   0.000000
NumberOfTimes90DaysLate                           0   0.000000
NumberRealEstateLoansOrLines                      0   0.000000
NumberOfTime60-89DaysPastDueNotWorse              0   0.000000
NumberOfDependents                             3924   2.616000


In [ ]:
df['NumberOfDependents_imputed'] = df['NumberOfDependents']. \
fillna(df['NumberOfDependents'].median())
df['MonthlyIncome_null_flag'] = df['MonthlyIncome'].isnull().astype(int)
df['MonthlyIncome_imputed'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())

In [ ]:
df

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,NumberOfDependents_imputed,MonthlyIncome_null_flag,MonthlyIncome_imputed
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0,2.0,0,9120.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0,1.0,0,2600.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0,0.0,0,3042.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0,0.0,0,3300.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0,0.0,0,63588.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,149996,0,0.040674,74,0,0.225131,2100.0,4,0,1,0,0.0,0.0,0,2100.0
149996,149997,0,0.299745,44,0,0.716562,5584.0,4,0,1,0,2.0,2.0,0,5584.0
149997,149998,0,0.246044,58,0,3870.000000,NaN,18,0,1,0,0.0,0.0,1,5400.0
149998,149999,0,0.000000,30,0,0.000000,5716.0,4,0,0,0,0.0,0.0,0,5716.0


In [ ]:
df.loc[df['age'] == 0, 'age'] = df['age'].median()

In [ ]:
late_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]

for col in late_cols:
    anomalous_count = (df[col] >= 96).sum()
    print(f"Аномальных значений (>=96) в {col}: {anomalous_count}")
    df[col] = df[col].apply(lambda x: 10 if x >= 10 else x)

Аномальных значений (>=96) в NumberOfTime30-59DaysPastDueNotWorse: 0
Аномальных значений (>=96) в NumberOfTime60-89DaysPastDueNotWorse: 0
Аномальных значений (>=96) в NumberOfTimes90DaysLate: 0


In [19]:
df[df['RevolvingUtilizationOfUnsecuredLines'] > 1]['RevolvingUtilizationOfUnsecuredLines']

,RevolvingUtilizationOfUnsecuredLines
162,1.046279
191,1.095083
226,1.953488
251,1.048211
293,2340.000000
...,...
149939,1.049900
149955,1.135552
149962,1.005733
149964,1.010934


In [38]:
border = df['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99)
df['RevolvingUtilization_clean'] = np.where(
    df['RevolvingUtilizationOfUnsecuredLines'] > border,
    border,
    df['RevolvingUtilizationOfUnsecuredLines']
)

## 2. **Feature Engineering, WoE/IV transformation**

In [20]:
df['EstimatedDebtPayment'] = df['DebtRatio'] * df['MonthlyIncome_imputed']

In [21]:
df['TotalPastDue'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

In [33]:
df['PastDuePerCreditLine'] = df['TotalPastDue'] / (df['NumberOfOpenCreditLinesAndLoans'] + 1)

In [29]:
def calculate_woe_iv(data, feature, target, bins=10):

    df_temp = data[[feature, target]].copy()

    if pd.api.types.is_numeric_dtype(df_temp[feature]):
        try:
            df_temp['bin'] = pd.qcut(df_temp[feature], q=bins, duplicates='drop')
        except ValueError:
            df_temp['bin'] = pd.cut(df_temp[feature], bins=bins)
    else:
        df_temp['bin'] = df_temp[feature]

    grouped = df_temp.groupby('bin', observed=False)[target].agg(
        total='count',
        bads='sum'
    ).reset_index()

    grouped['goods'] = grouped['total'] - grouped['bads']

    total_goods = grouped['goods'].sum()
    total_bads = grouped['bads'].sum()

    grouped['pct_goods'] = grouped['goods'] / total_goods
    grouped['pct_bads'] = grouped['bads'] / total_bads

    grouped['pct_goods'] = grouped['pct_goods'].replace(0, 0.0001)
    grouped['pct_bads'] = grouped['pct_bads'].replace(0, 0.0001)

    grouped['WoE'] = np.log(grouped['pct_goods'] / grouped['pct_bads'])
    grouped['IV'] = (grouped['pct_goods'] - grouped['pct_bads']) * grouped['WoE']

    iv_val = grouped['IV'].sum()

    return grouped, iv_val

In [41]:
target_col = 'SeriousDlqin2yrs'
features_to_eval = [
    'age', 'RevolvingUtilization_clean', 'MonthlyIncome_imputed',
    'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate', 'DebtRatio', 'NumberOfOpenCreditLinesAndLoans',
    'NumberRealEstateLoansOrLines', 'NumberOfDependents_imputed',
    'TotalPastDue', 'PastDuePerCreditLine'
]

iv_summary = []

for feat in features_to_eval:
    _, iv = calculate_woe_iv(df, feat, target_col, bins=10)
    iv_summary.append({'Feature': feat, 'IV': iv})

iv_df = pd.DataFrame(iv_summary).sort_values(by='IV', ascending=False)

In [44]:
print("\n--- Сводка Information Value (IV) по фичам ---")
print(iv_df.to_string(index=False))


--- Сводка Information Value (IV) по фичам ---
                             Feature       IV
                PastDuePerCreditLine 1.353439
          RevolvingUtilization_clean 1.113378
                        TotalPastDue 1.050441
NumberOfTime30-59DaysPastDueNotWorse 0.471831
                                 age 0.259158
                           DebtRatio 0.073666
               MonthlyIncome_imputed 0.066954
     NumberOfOpenCreditLinesAndLoans 0.066892
          NumberOfDependents_imputed 0.024965
        NumberRealEstateLoansOrLines 0.012091
NumberOfTime60-89DaysPastDueNotWorse 0.000000
             NumberOfTimes90DaysLate 0.000000


## 3. **PD-model, scoreboard**

In [45]:
selected_features = [
    'RevolvingUtilization_clean',
    'TotalPastDue',
    'PastDuePerCreditLine',
    'age',
    'DebtRatio',
    'MonthlyIncome_imputed'
]

In [46]:
X = df[selected_features]
y = df['SeriousDlqin2yrs']

In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [53]:
class WoEEncoder:
    def __init__(self, bins=5):
        self.bins = bins
        self.woe_maps = {}

    def fit(self, X, y):
        df_tr = X.copy()
        df_tr['target'] = y.values

        for col in X.columns:

            if pd.api.types.is_numeric_dtype(df_tr[col]):
                try:
                    binned = pd.qcut(df_tr[col], q=self.bins, duplicates='drop')
                except ValueError:
                    binned = pd.cut(df_tr[col], bins=self.bins)
            else:
                binned = df_tr[col]

            grouped = df_tr.groupby(binned, observed=False)['target'].agg(['count', 'sum'])
            grouped.columns = ['total', 'bads']
            grouped['goods'] = grouped['total'] - grouped['bads']

            grouped['goods'] = grouped['goods'].replace(0, 0.5)
            grouped['bads'] = grouped['bads'].replace(0, 0.5)

            tot_goods = (y == 0).sum()
            tot_bads = (y == 1).sum()

            grouped['pct_goods'] = grouped['goods'] / tot_goods
            grouped['pct_bads'] = grouped['bads'] / tot_bads

            grouped['WoE'] = np.log(grouped['pct_goods'] / grouped['pct_bads'])
            self.woe_maps[col] = grouped['WoE'].to_dict()

        return self

    def transform(self, X):
        X_out = pd.DataFrame(index=X.index)
        for col in X.columns:
            if pd.api.types.is_numeric_dtype(X[col]):
                try:
                    binned = pd.qcut(X[col], q=self.bins, duplicates='drop')
                except ValueError:
                    binned = pd.cut(X[col], bins=self.bins)
            else:
                binned = X[col]

            X_out[col] = binned.map(self.woe_maps[col]).astype(float)
        return X_out

In [78]:
encoder = WoEEncoder(bins=5)
encoder.fit(X_train, y_train)

X_train_woe = encoder.transform(X_train)
X_test_woe = encoder.transform(X_test)
X_train_woe = X_train_woe.fillna(0)
X_test_woe = X_test_woe.fillna(0)

In [79]:
model = LogisticRegression(random_state=42)
model.fit(X_train_woe, y_train)

LogisticRegression(random_state=42)

In [71]:
X_train_woe

,RevolvingUtilization_clean,TotalPastDue,PastDuePerCreditLine,age,DebtRatio,MonthlyIncome_imputed
57836,0.873304,0.561199,0.923843,0.413468,0.214842,0.193507
132895,1.296768,0.561199,0.923843,1.042231,-0.430662,-0.183738
27981,0.873304,0.561199,0.923843,-0.480963,0.165125,-0.183738
37852,-1.244476,0.561199,0.923843,0.413468,0.214842,0.193507
103813,0.873304,0.561199,0.923843,0.413468,-0.430662,-0.342676
...,...,...,...,...,...,...
18048,1.296768,0.561199,0.923843,-0.240230,-0.430662,0.104324
3895,1.296768,0.561199,0.923843,-0.083546,0.091165,0.377705
109980,-1.244476,0.561199,0.923843,1.042231,0.214842,0.193507
74354,-1.244476,0.561199,-1.394220,0.413468,0.091165,-0.342676


In [72]:
TARGET_SCORE = 600
TARGET_ODDS = 50
PDO = 20

Factor = PDO / np.log(2)
Offset = TARGET_SCORE - (Factor * np.log(TARGET_ODDS))

In [73]:
n_feats = len(selected_features)
intercept = model.intercept_[0]

scorecard_rows = []

for feat in selected_features:
    coef = model.coef_[0][selected_features.index(feat)]
    woe_dict = encoder.woe_maps[feat]

    for bin_interval, woe_val in woe_dict.items():
        score_point = - (woe_val * coef + intercept / n_feats) * Factor + (Offset / n_feats)
        scorecard_rows.append({
            'Признак': feat,
            'Интервал (Bin)': str(bin_interval),
            'WoE': round(woe_val, 4),
            'Баллы (Score)': int(round(score_point))
        })

scorecard = pd.DataFrame(scorecard_rows)

print("--- ФРАГМЕНТ СКОРИНГОВОЙ КАРТЫ (SCORECARD) ---")
print(scorecard.to_string(index=False))

--- ФРАГМЕНТ СКОРИНГОВОЙ КАРТЫ (SCORECARD) ---
                   Признак      Интервал (Bin)     WoE  Баллы (Score)
RevolvingUtilization_clean    (-0.001, 0.0191]  1.2968            118
RevolvingUtilization_clean    (0.0191, 0.0826]  1.4137            120
RevolvingUtilization_clean     (0.0826, 0.271]  0.8733            110
RevolvingUtilization_clean      (0.271, 0.696] -0.0451             93
RevolvingUtilization_clean      (0.696, 1.093] -1.2445             71
              TotalPastDue       (-0.001, 1.0]  0.5612            101
              TotalPastDue         (1.0, 30.0] -2.0328             69
      PastDuePerCreditLine    (-0.001, 0.0435]  0.9238            108
      PastDuePerCreditLine      (0.0435, 30.0] -1.3942             73
                       age      (-0.001, 39.0] -0.4810             88
                       age        (39.0, 48.0] -0.2402             91
                       age        (48.0, 56.0] -0.0835             93
                       age        (56.0, 65

## 4. **Scoreboard rating**

In [81]:
y_pred_test_pd = model.predict_proba(X_test_woe)[:, 1]

In [82]:
auc_test = roc_auc_score(y_test, y_pred_test_pd)
gini_test = 2 * auc_test - 1

In [83]:
print(f"ROC-AUC: {auc_test:.4f}")
print(f"Gini:    {gini_test:.4f}")

ROC-AUC: 0.7569
Gini:    0.5138
